### ***Advanced Micro Devices***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/Advanced Micro Devices/AMD.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_amd_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze AMD stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze AMD stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 192b4e96-0eba-445b-ac19-b5f557e7fa9c, 10, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.
Bronze AMD stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, 548b5600-69d8-4924-aebd-28feefcc842c)

### ***Amazon***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/Amazon/AMZN.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_amzn_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze AMZN stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze AMZN stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 192b4e96-0eba-445b-ac19-b5f557e7fa9c, 8, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.


Bronze AMZN stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, 6779d18a-3f14-40a6-8beb-bf2361e2a1ab)

### ***Apple***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/Apple/AAPL.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_aapl_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze AAPL stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze AAPL stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 192b4e96-0eba-445b-ac19-b5f557e7fa9c, 6, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.


Bronze AAPL stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, dd043512-a0b9-459f-893f-9ac8e73e7d92)

### ***Intel***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/Intel/INTC.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_intc_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze INTC stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze INTC stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 192b4e96-0eba-445b-ac19-b5f557e7fa9c, 4, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.


Bronze INTC stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, 93c09d79-55ab-45a2-b5e6-93a791535a23)

### ***JPMorgan Chase***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/JPMorgan Chase/JPM.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_jpm_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze JPM stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze JPM stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 223223a7-63f0-4ae9-8d96-0594f9813d8a, 32, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.


Bronze JPM stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, b0c1f6eb-a1a7-4282-8bba-12cdb5929ece)

### ***Meta Platforms***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/Meta Platforms/META.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_meta_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze META stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze META stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 223223a7-63f0-4ae9-8d96-0594f9813d8a, 30, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.


Bronze META stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, ca2b9c1b-5692-49de-bb29-fe435f9e83db)

### ***Microsoft***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/Microsoft/MSFT.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_msft_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze MSFT stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze MSFT stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 223223a7-63f0-4ae9-8d96-0594f9813d8a, 28, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.


Bronze MSFT stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, 3c63809e-44bf-468d-8b81-8ebce216fff5)

### ***Netflix***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/Netflix/NFLX.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_nflx_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze NFLX stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze NFLX stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 223223a7-63f0-4ae9-8d96-0594f9813d8a, 26, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.


Bronze NFLX stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, 2bb55cf3-0a8c-4dc7-8087-2e48b4258873)

### ***Nvidia***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/Nvidia/NVDA.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_nvda_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze NVDA stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze NVDA stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 223223a7-63f0-4ae9-8d96-0594f9813d8a, 23, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.


Bronze NVDA stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, 3db17dc8-c97e-4268-acc6-e46430392d6a)

### ***Oracle***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/Oracle/ORCL.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_orcl_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze ORCL stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze ORCL stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 223223a7-63f0-4ae9-8d96-0594f9813d8a, 21, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.


Bronze ORCL stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, 0791f020-ee31-4d28-b97a-7cee744a4f43)

### ***Palantir Technologies***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/Palantir Technologies/PLTR.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_pltr_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze PLTR stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze PLTR stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 223223a7-63f0-4ae9-8d96-0594f9813d8a, 18, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.


Bronze PLTR stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, ae469495-41fc-440b-85ae-b8217b3b8fbb)

### ***PayPal***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/PayPal/PYPL.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_pypl_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze PYPL stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze PYPL stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 223223a7-63f0-4ae9-8d96-0594f9813d8a, 16, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.


Bronze PYPL stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, 118366c3-62f2-4905-a73b-f9da2f601d3f)

### ***Salesforce***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/Salesforce/CRM.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_crm_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze CRM stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze CRM stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 223223a7-63f0-4ae9-8d96-0594f9813d8a, 14, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.


Bronze CRM stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, 74e65267-c774-4ba9-9193-1c5d86682d7f)

### ***Tesla***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/Tesla/TSLA.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_tsla_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze TSLA stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze TSLA stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 223223a7-63f0-4ae9-8d96-0594f9813d8a, 10, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.


Bronze TSLA stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, 5e1a7290-3400-4566-8e91-b63f81f423d2)

### ***Walmart***

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import *

raw_path=f"Files/Stocks_Raw_Data/Walmart/WMT.json"
schema='bronze'

#Reading and loading the json data into a Dataframe
df = spark.read.option("multiline", "true").json(raw_path)

# Extract the stock result and expand its array
stock_df = df.select(
    explode(col("chart.result")).alias("stock")
)

# Extract metadata and the arrays containing daily prices
price_df = stock_df.select(
    col("stock.meta.symbol").alias("ticker"),
    col("stock.meta.currency").alias("currency"),
    col("stock.timestamp").alias("timestamps"),
    col("stock.indicators.quote")[0]["open"].alias("opens"),
    col("stock.indicators.quote")[0]["high"].alias("highs"),
    col("stock.indicators.quote")[0]["low"].alias("lows"),
    col("stock.indicators.quote")[0]["close"].alias("closes"),
    col("stock.indicators.quote")[0]["volume"].alias("volumes"),
    col("stock.indicators.adjclose")[0]["adjclose"].alias("adj_closes")
)

# Match each timestamp with its corresponding price values
price_df = price_df.withColumn(
    "daily",
    explode(
        arrays_zip(
            "timestamps",
            "opens",
            "highs",
            "lows",
            "closes",
            "volumes",
            "adj_closes"
        )
    )
)

# Create a tabular DataFrame
df_bronze = price_df.select(
    "ticker",
    "currency",
    col("daily.timestamps").alias("timestamp"),
    to_timestamp(
        col("daily.timestamps").cast("long")
    ).alias("trade_datetime"),
    col("daily.opens").alias("open"),
    col("daily.highs").alias("high"),
    col("daily.lows").alias("low"),
    col("daily.closes").alias("close"),
    col("daily.volumes").alias("volume"),
    col("daily.adj_closes").alias("adj_close")
)

# Add the ingestion timestamp and source file path columns for data lineage and reference
df_bronze=df_bronze.withColumn("ingested_at", current_timestamp()) \
    .withColumn("file_path", input_file_name())

# WRITE MECHANISM: DELTA TABLE + SCD TYPE 2
table_name = f"FABRIC_COE_WORKLOAD.stock_market_analysis_LH.{schema}.brz_wmt_stock"
primary_key = ["ticker", "timestamp"]

# Prepare source DataFrame with SCD Type 2 tracking columns
df_source = (
    df_bronze
    .dropDuplicates(primary_key)
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

df_source.createOrReplaceTempView("vw_stock_source")

# Check whether the Delta table already exists
if not spark.catalog.tableExists(table_name):

    # Initial load: create the Delta table
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .option("mergeSchema", True) \
        .saveAsTable(table_name)

    print("Bronze WMT stock Delta table created successfully")

else:
    # Existing table: process SCD Type 2 changes
    print("Table already exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values have changed
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_stock_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new business keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_stock_versions AS
        SELECT source.*
        FROM vw_stock_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_stock_versions
    """)

    print("Bronze WMT stock SCD Type 2 processing completed")

# Display the complete table, including SCD Type 2 history
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)


StatementMeta(, 223223a7-63f0-4ae9-8d96-0594f9813d8a, 8, Finished, Available, Finished, False)

Table already exists. Applying SCD Type 2 logic.


Bronze WMT stock SCD Type 2 processing completed


SynapseWidget(Synapse.DataFrame, 85a76545-5258-42c9-b431-11d4e64a3552)

### ***Bronze table data simple Verification***

In [1]:
from pyspark.sql.functions import col

catalog_db = "FABRIC_COE_WORKLOAD.stock_market_analysis_LH"

bronze_tables = [
    "brz_aapl_stock", "brz_amd_stock", "brz_amzn_stock",
    "brz_crm_stock", "brz_intc_stock", "brz_jpm_stock",
    "brz_meta_stock", "brz_msft_stock", "brz_nflx_stock",
    "brz_nvda_stock", "brz_orcl_stock", "brz_pltr_stock",
    "brz_pypl_stock", "brz_tsla_stock", "brz_wmt_stock"
]

for table in bronze_tables:
    df = spark.table(f"{catalog_db}.bronze.{table}")

    print(f"\n--- {table} ---")

    display(
        df.groupBy("ticker", "file_path")
          .count()
          .orderBy("ticker", "file_path")
    )

StatementMeta(, d78f91df-6879-46d5-b678-b73c332310a9, 3, Finished, Available, Finished, False)


--- brz_aapl_stock ---


SynapseWidget(Synapse.DataFrame, 72cc4a9f-d490-4f3f-8d34-23144394e84a)


--- brz_amd_stock ---


SynapseWidget(Synapse.DataFrame, 20482ee5-fde0-4d9f-9ab4-8a252f6cc999)

...

SynapseWidget(Synapse.DataFrame, 452e5872-827f-46ac-a1c7-6de4927cf12c)


--- brz_tsla_stock ---


SynapseWidget(Synapse.DataFrame, c97b6fcb-6492-42be-8860-054751eed176)


--- brz_wmt_stock ---


SynapseWidget(Synapse.DataFrame, 983521b2-a128-42e6-b959-ac9f7a09006d)